# HackerNews Analytics Pro — v11 Walk-Forward Research

This notebook evaluates the v10 classifiers over multiple expanding chronological windows and adds probability-calibration diagnostics. The bundled dataset is synthetic; use a larger real dataset for research conclusions.

## 1. Load data and inspect time coverage

In [ ]:
from hn_data_loader import load_stories
from hn_walkforward_v11 import walk_forward_benchmark, export_v11_walkforward

data = load_stories('../demo_stories.jsonl')
data.shape, data[['time', 'score']].describe()

## 2. Run expanding-window evaluation

Each fold trains on all earlier observations and tests on the next chronological block. The relative high-score threshold is recomputed from that fold's training scores only.

In [ ]:
summary, fold_metrics, predictions, metadata = walk_forward_benchmark(
    data, n_splits=3, test_fraction=0.15, bootstrap_repeats=500
)
display(summary)
display(fold_metrics)

## 3. Compare ranking and calibration quality

In [ ]:
import matplotlib.pyplot as plt

summary.set_index('model')[['roc_auc_mean', 'average_precision_mean']].plot(kind='bar', figsize=(9, 4), title='Walk-forward ranking metrics')
plt.ylabel('Mean across folds'); plt.tight_layout()

summary.set_index('model')[['brier_score_mean', 'log_loss_mean', 'expected_calibration_error_mean']].plot(kind='bar', figsize=(10, 4), title='Probability calibration diagnostics')
plt.ylabel('Lower is better'); plt.tight_layout()

## 4. Inspect out-of-fold predictions

Every row is a prediction made for a later fold by a model trained only on earlier rows. Brier score and log loss assess probability quality; ECE is a coarse equal-width-bin diagnostic. No post-hoc probability calibration is fitted.

In [ ]:
display(predictions.head(20))
display(predictions.groupby('model').agg(rows=('actual_high_score', 'size'), mean_probability=('high_score_probability', 'mean'), observed_rate=('actual_high_score', 'mean')))

## 5. Export reproducible artifacts

In [ ]:
result = export_v11_walkforward(data, output_dir='../reports', n_splits=3, test_fraction=0.15, bootstrap_repeats=500)
result

## Interpretation and limitations

- A fold can contain only one target class; ROC-AUC and average precision are undefined for that fold and are omitted from fold averages.
- The target is relative to the training-period score quantile, not a fixed future score threshold.
- Bootstrap intervals describe pooled out-of-fold discrimination uncertainty under row resampling; they do not account for every temporal dependence.
- These are observational, retrospective diagnostics—not causal estimates or guaranteed predictions. The demo data is synthetic.